# TP1 — Building a ReAct Agent from Scratch
## Module 2IAiasd 9.7 — Agentic AI — IMT Mines Alès

**Prerequisites:** You have completed TD1 (Exercises 0–4).  
Your Ollama server is still running and `qwen2.5:3b` is available.

**Objective:** Implement the full **ReAct loop** (Reason + Act) introduced in CM1,  
using the tools and concepts you practised in TD1.  
By the end of this session you will have a working agent that solves multi-step problems autonomously.



---

### Structure

| # | Exercise | Concepts |
|---|---------|----------|
| 1 | The ReAct loop — step by step | orchestrator, tool dispatch, ToolMessage |
| 2 | A complete math agent | multi-step reasoning, safety limit |
| 3 | Mixed-tool agent | combining tool types, robustness |
| 4 | Your own agent (open-ended) | design + evaluation |

> 📖 **Reference:** CM1 slides — perception → reasoning → action → observation loop  
> 📖 **Reference:** TD1 Exercise 4 — `@tool`, `bind_tools`, `ToolMessage`  


---
## Setup — Carry over from TD1

Run this cell to restore the TD1 environment.  
Make sure `ollama serve` is still running in your terminal.

In [1]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
import requests, os

MODEL = "qwen2.5:3b"

# Quick health check
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=3)
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"✅ Ollama running | available models: {models}")
except Exception:
    print("❌ Ollama not reachable — run: ollama serve")

llm = ChatOllama(model=MODEL, temperature=0)
print(f"✅ LLM ready: {MODEL}")

# Optional: LangSmith tracing (from TD1)
# os.environ["LANGSMITH_API_KEY"]    = "lsv2_pt_..."
# os.environ["LANGSMITH_TRACING_V2"] = "true"
# os.environ["LANGSMITH_PROJECT"]    = "2IAiasd-9.7-TP1"

✅ Ollama running | available models: ['qwen2.5:3b', 'llama3.1:latest', 'nomic-embed-text:latest']
✅ LLM ready: qwen2.5:3b


---
## Exercise 1 — The ReAct Loop, Step by Step

In CM1 you saw the agent loop conceptually:
```
Perceive → Reason (LLM) → Act (tool call) → Observe (tool result) → loop
```

In TD1 Exercise 4 you made **one** tool call manually.  
Now you build the **full loop** that keeps running until the LLM decides it is done.

### 1.1 — Trace what the loop sees at each step

In [3]:
# Tools from TD1 Exercise 4 — reload them here
@tool
def get_word_count(text: str) -> int:
    """Count the number of words in a given text string."""
    return len(text.split())

@tool
def celsius_to_fahrenheit(celsius: float) -> float:
    """Convert a temperature value from Celsius to Fahrenheit."""
    return round(celsius * 9 / 5 + 32, 2)

@tool
def get_current_date() -> str:
    """Return today's date in ISO format YYYY-MM-DD."""
    from datetime import date
    return date.today().isoformat()

@tool
def calculate_bmi(weight_kg: float, height_m: float) -> float:
    """Calculate Body Mass Index given weight in kilograms and height in metres."""
    return round(weight_kg / (height_m ** 2), 2)

tools_ex1 = [get_word_count, celsius_to_fahrenheit, get_current_date, calculate_bmi]
tools_map_ex1 = {t.name: t for t in tools_ex1}

llm_with_tools = llm.bind_tools(tools_ex1)
print(f"Available tools: {[t.name for t in tools_ex1]}")

Available tools: ['get_word_count', 'celsius_to_fahrenheit', 'get_current_date', 'calculate_bmi']


In [5]:
# Trace exactly what happens at each step for the query:
# "What is today's date and how many words are in 'Hello from IMT Mines Ales in France'?"

messages = [
    SystemMessage(content="You are a helpful assistant. Use tools when needed."),
    HumanMessage(content="What is today's date and how many words are in 'Hello from IMT Mines Ales in France'?"),
]

print("=" * 60)
print(f"INITIAL STATE: {len(messages)} messages")
for m in messages:
    print(f"  {m.type}: {m.content[:60]}")
print("=" * 60)

step = 0
while True:
    step += 1
    print(f"\n--- STEP {step}: REASON ---")

    response = llm_with_tools.invoke(messages)
    messages.append(response)

    print(f"  LLM content : {repr(response.content[:80]) if response.content else '(empty)'}")
    print(f"  Tool calls  : {response.tool_calls}")
    print(f"  Msg count   : {len(messages)}")

    if not response.tool_calls:
        print("\n--- LOOP EXITS: no more tool calls ---")
        break

    print(f"\n--- STEP {step}: ACT + OBSERVE ---")
    for tc in response.tool_calls:
        fn = tools_map_ex1[tc["name"]]
        result = fn.invoke(tc["args"])
        tool_msg = ToolMessage(content=str(result), tool_call_id=tc["id"])
        messages.append(tool_msg)
        print(f"  {tc['name']}({tc['args']}) → {result}")

print(f"\n{'=' * 60}")
print(f"FINAL ANSWER: {response.content}")
print(f"Total steps: {step} | Total messages: {len(messages)}")

INITIAL STATE: 2 messages
  system: You are a helpful assistant. Use tools when needed.
  human: What is today's date and how many words are in 'Hello from I

--- STEP 1: REASON ---
  LLM content : (empty)
  Tool calls  : [{'name': 'get_current_date', 'args': {}, 'id': '843d9ce9-ca80-4dd4-890a-aa3eabbd3a43', 'type': 'tool_call'}, {'name': 'get_word_count', 'args': {'text': 'Hello from IMT Mines Ales in France'}, 'id': '51dc19ca-4dee-4397-b75f-0c801986cd86', 'type': 'tool_call'}]
  Msg count   : 3

--- STEP 1: ACT + OBSERVE ---
  get_current_date({}) → 2026-10-17
  get_word_count({'text': 'Hello from IMT Mines Ales in France'}) → 7

--- STEP 2: REASON ---
  LLM content : "Today's date is 2026-10-17 and the sentence 'Hello from IMT Mines Ales in France"
  Tool calls  : []
  Msg count   : 6

--- LOOP EXITS: no more tool calls ---

FINAL ANSWER: Today's date is 2026-10-17 and the sentence 'Hello from IMT Mines Ales in France' contains 7 words.
Total steps: 2 | Total messages: 6


### 1.2 — ✏️ Your turn: trace analysis

Answer the following questions **in the Markdown cell below**,  
based on the trace you just observed. Be precise and reference specific message types.

**Answers (replace this text):**

1. How many tool calls did the model make? Which tools were called and in what order?
   > *two tools :  get_current_date and get_words_counts in order*

2. What does `response.content` contain when the model decides to call a tool vs. when it gives the final answer?
   > *it's empty when the model call the tools because the answers are in response.tool_calls.*

3. What is the stopping condition of the loop? Why is it reliable?
   > *when the llm doesn't need any more tools*

4. What would happen if you removed the `SystemMessage`? Try it and report.
   > *less presice answers*

---
## Exercise 2 — A Complete Math Agent

Now you build a **reusable `react_agent` function** that cleanly encapsulates the loop.  
This is exactly the pattern used by every production agent framework (LangGraph, CrewAI, AutoGen…).

### 2.1 — Define the math tools

In [6]:
@tool
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b

@tool
def subtract(a: float, b: float) -> float:
    """Subtract b from a."""
    return a - b

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

@tool
def divide(a: float, b: float) -> float:
    """Divide a by b. Raises ValueError if b is zero."""
    if b == 0:
        raise ValueError("Cannot divide by zero.")
    return round(a / b, 6)

@tool
def power(base: float, exponent: float) -> float:
    """Raise base to the power of exponent."""
    return base ** exponent

@tool
def square_root(x: float) -> float:
    """Return the square root of x. Raises ValueError if x is negative."""
    import math
    if x < 0:
        raise ValueError("Cannot compute the square root of a negative number.")
    return round(math.sqrt(x), 6)

math_tools = [add, subtract, multiply, divide, power, square_root]
print(f"Math tools: {[t.name for t in math_tools]}")

Math tools: ['add', 'subtract', 'multiply', 'divide', 'power', 'square_root']


### 2.2 — ✏️ Implement `react_agent`

Complete the function below. Requirements:
- Accepts `query` (str), `tools` (list), `system_prompt` (str, optional), `max_steps` (int, default 10), `verbose` (bool, default True)
- Returns the final answer as a string
- In verbose mode: prints each step's tool calls and observations
- Handles tool execution errors gracefully (catch exceptions, pass the error message as a `ToolMessage`)
- Stops if `max_steps` is reached and returns a timeout message

In [8]:
def react_agent(
    query: str,
    tools: list,
    system_prompt: str = "You are a helpful assistant. Use tools when needed to answer accurately.",
    max_steps: int = 10,
    verbose: bool = True,
) -> str:
    # 1. Messages initiaux
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=query),
    ]

    # 2. Bind des outils
    llm_with_tools = llm.bind_tools(tools)

    # 3. Dictionnaire nom -> outil
    tools_map = {t.name: t for t in tools}

    # 4. Boucle ReAct, limitée à max_steps
    for step in range(1, max_steps + 1):
        # REASON
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if verbose:
            print(f"\n--- STEP {step} ---")
            print(f"  Tool calls: {response.tool_calls}")

        # Condition d'arrêt : plus d'appel d'outil -> réponse finale
        if not response.tool_calls:
            return response.content

        # ACT + OBSERVE
        for tc in response.tool_calls:
            # 5. Gestion des erreurs
            try:
                if tc["name"] not in tools_map:
                    raise KeyError(f"Unknown tool: {tc['name']}")
                result = tools_map[tc["name"]].invoke(tc["args"])
            except Exception as e:
                result = f"Error: {e}"

            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

            if verbose:
                print(f"  {tc['name']}({tc['args']}) -> {result}")

    # 6. max_steps atteint sans réponse finale
    return f"Timeout: no final answer after {max_steps} steps."

In [11]:
# Test 1: simple two-step calculation
result = react_agent("What is (3 + 5) * 12?", math_tools)
print(f"\nResult: {result}")
if "96" not in result:
    # Retry with an explicit tool-use instruction for small models
    result = react_agent(
    "What is (3 + 5) * 12?",
    math_tools,
    system_prompt=(
        "You are a math assistant. Always use the provided tools for every arithmetic "
        "operation. Call only one tool at a time, wait for its result, then continue. "
        "When you have the final number, state it clearly."
    ),
)

print(result)


--- STEP 1 ---
  Tool calls: [{'name': 'add', 'args': {'b': 5, 'a': 3}, 'id': '00943266-596a-4519-bda9-70e2ad581bd8', 'type': 'tool_call'}, {'name': 'multiply', 'args': {'b': 'result_of_addition', 'a': 12}, 'id': '1de379ef-0285-4dfd-892a-9d05f302f604', 'type': 'tool_call'}]
  add({'b': 5, 'a': 3}) -> 8.0
  multiply({'b': 'result_of_addition', 'a': 12}) -> Error: 1 validation error for multiply
b
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='result_of_addition', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/float_parsing

--- STEP 2 ---
  Tool calls: [{'name': 'add', 'args': {'a': 3, 'b': 5}, 'id': '36490718-dde5-450f-aaca-c0a75963a44f', 'type': 'tool_call'}, {'name': 'multiply', 'args': {'a': 12, 'b': 'result_of_addition'}, 'id': 'b1e4c0ea-643c-409b-a052-e77aeebb9a6d', 'type': 'tool_call'}]
  add({'a': 3, 'b': 5}) -> 8.0
  multiply({'a': 12, 'b': 'result_of_addition'}) -> Error: 1 validatio

In [12]:
# Test 2: multi-step with power and square root
result = react_agent(
    "Calculate: 2 to the power of 8, then subtract 56, then take the square root.",
    math_tools
)
print(f"\nResult: {result}")
# 2^8 = 256, 256 - 56 = 200, sqrt(200) ≈ 14.142
assert "14" in result, "Expected ~14.14 in the answer"
print("✅ Test 2 passed")


--- STEP 1 ---
  Tool calls: [{'name': 'power', 'args': {'base': 2, 'exponent': 8}, 'id': 'ba7172ee-637d-4d67-9d7a-92f026e62d4f', 'type': 'tool_call'}]
  power({'base': 2, 'exponent': 8}) -> 256.0

--- STEP 2 ---
  Tool calls: [{'name': 'subtract', 'args': {'a': 256, 'b': 56}, 'id': 'e47112eb-dd2f-425b-8f06-55fd556ac254', 'type': 'tool_call'}]
  subtract({'a': 256, 'b': 56}) -> 200.0

--- STEP 3 ---
  Tool calls: [{'name': 'square_root', 'args': {'x': 200}, 'id': 'a1a3e1ce-f224-43f3-ad4a-e97f945487b1', 'type': 'tool_call'}]
  square_root({'x': 200}) -> 14.142136

--- STEP 4 ---
  Tool calls: []

Result: The final result after performing the operations is approximately 14.142136.
✅ Test 2 passed


In [13]:
# Test 3: error handling — division by zero
result = react_agent("What is 42 divided by 0?", math_tools, verbose=True)
print(f"\nResult: {result}")
# The agent should receive the error and report it gracefully without crashing
print("✅ Test 3 — verify the agent handled the error without raising an exception")


--- STEP 1 ---
  Tool calls: [{'name': 'divide', 'args': {'a': 42, 'b': 0}, 'id': '1dba92cb-1e22-487e-aa6c-25a42546c1b6', 'type': 'tool_call'}]
  divide({'a': 42, 'b': 0}) -> Error: Cannot divide by zero.

--- STEP 2 ---
  Tool calls: []

Result: The operation of dividing 42 by 0 is not valid in mathematics because division by zero is undefined. Therefore, I cannot compute the result.
✅ Test 3 — verify the agent handled the error without raising an exception


In [14]:
# Test 4: no tool needed
result = react_agent("What is the Pythagorean theorem?", math_tools, verbose=False)
print(f"Result: {result}")
print("✅ Test 4 — agent answered without calling any tool")

Result: The Pythagorean theorem states that in a right-angled triangle, the square of the length of the hypotenuse (the side opposite the right angle) is equal to the sum of the squares of the lengths of the other two sides. It can be written as: a² + b² = c², where c represents the length of the hypotenuse, and a and b represent the lengths of the other two sides.

I can help you perform calculations related to this theorem, if you need. For example, if you provide me with the lengths of two sides, I can calculate the length of the third side. Would you like to try that?
✅ Test 4 — agent answered without calling any tool


---
## Exercise 3 — Mixed-Tool Agent

A real-world agent needs tools of **different types** — not just one category.  
Here you combine the math tools from Exercise 2 with the utility tools from TD1,  
and add two new tools of your own design.

### 3.1 — Combine the existing tools

In [15]:
# Bring back the utility tools from TD1
combined_tools = math_tools + [get_word_count, celsius_to_fahrenheit, get_current_date, calculate_bmi]
print(f"Combined tools ({len(combined_tools)}): {[t.name for t in combined_tools]}")

Combined tools (10): ['add', 'subtract', 'multiply', 'divide', 'power', 'square_root', 'get_word_count', 'celsius_to_fahrenheit', 'get_current_date', 'calculate_bmi']


### 3.2 — ✏️ Add 2 new tools of your choice

Design and implement 2 new tools. Pick from the suggestions below or invent your own.  
Each tool must have:
- A clear, precise **docstring** (remember: the docstring IS the prompt sent to the LLM)
- **Type annotations** on all parameters
- At least one **edge case** handled (empty input, division by zero, negative value…)

**Suggestions:** `reverse_string`, `is_palindrome`, `count_vowels`, `days_until_date`, `percentage(value, total)`, `fibonacci(n)`, `is_prime(n)`

In [ ]:
@tool
def count_vowels(text :string)->int :
    """TODO: count the """
    pass  # implement

@tool
def my_tool_b(...):
    """TODO: write a clear docstring that tells the LLM exactly when and how to use this tool."""
    pass  # implement

# Add your tools to the combined list
all_tools = combined_tools + [my_tool_a, my_tool_b]
print(f"All tools ({len(all_tools)}): {[t.name for t in all_tools]}")

### 3.3 — ✏️ Test with cross-category queries

Write **3 queries** that each require tools from **at least 2 different categories**  
(e.g. math + date, or string + math, or your custom tool + an existing one).

In [ ]:
# Query 1: requires math + date (or math + another category)
# Example: "What is today's date, and what is 2 to the power of the day number?"
# TODO: write your own query
query1 = "TODO: replace with your own cross-category query"
result1 = react_agent(query1, all_tools)
print(f"Q1: {query1}")
print(f"A1: {result1}\n")

In [ ]:
# Query 2: requires one of your custom tools + an existing tool
# TODO: write your own query
query2 = "TODO: replace with your own cross-category query"
result2 = react_agent(query2, all_tools)
print(f"Q2: {query2}")
print(f"A2: {result2}\n")

In [ ]:
# Query 3: a more complex chain (3+ tool calls expected)
# TODO: write your own query
query3 = "TODO: replace with your own multi-step, cross-category query"
result3 = react_agent(query3, all_tools)
print(f"Q3: {query3}")
print(f"A3: {result3}\n")

### 3.4 — ✏️ Reflection

**Reflection (replace this text):**

1. Did the agent always call tools in the order you expected? If not, why?
   > *Your answer:*

2. Did the model use your custom tools correctly? What role did the docstring play?
   > *Your answer:*

3. Did any query cause a wrong tool call or a wrong answer? If so, how would you fix it?
   > *Your answer:*

---
## Exercise 4 — Your Own Agent

Design and build a **domain-specific agent** of your choice.

**What to include:**
- At least **4 tools**, at least **2 original** (not reused from previous exercises)
- A carefully designed **system prompt**: role, constraints, output format
- At least **5 test queries**: normal cases, edge cases, and one no-tool case
- A brief **evaluation** of what worked and what didn't

### Domain ideas
- **Study assistant** — grade conversion (French 0–20 ↔ US A-F), GPA calculation, study-time estimator
- **Unit converter** — lengths, weights, temperatures, with context-aware explanations
- **Health tracker** — BMI, calorie estimation, heart-rate zones, VO2 max
- **Finance assistant** — interest calculation, savings projection, budget breakdown
- **Text analyser** — word count, reading time, keyword frequency, readability score
- **Your own idea** — describe it clearly in the design cell below

### 4.1 — ✏️ Agent design

Before writing any code, describe your agent here.

**Agent design (replace this text):**

**Name:**

**Domain:**

**Who would use it and for what?**
> *Your answer:*

**List of tools (name + one-sentence description each):**
1. `tool_name_1` —
2. `tool_name_2` —
3. `tool_name_3` —
4. `tool_name_4` —
(add more if needed)

**System prompt (draft):**
> *Your draft here:*

**5 test queries you plan to run:**
1. (normal case)
2. (multi-step — requires 2+ tool calls)
3. (edge case — error or boundary value)
4. (no tool needed — tests whether the agent resists unnecessary calls)
5. (your choice)

### 4.2 — ✏️ Implement the tools

In [ ]:
# TODO: define your tools here
# Each tool must have:
#   - a descriptive name
#   - a clear docstring (this is what the LLM reads to decide when to call it)
#   - type annotations on all parameters
#   - edge-case handling

@tool
def tool_1(...):
    """Precise description of what this tool does, when to use it, and what it returns."""
    pass

# Add all your tools here...

my_agent_tools = [tool_1]  # replace with your full list

### 4.3 — ✏️ Run the 5 test queries

In [ ]:
MY_SYSTEM_PROMPT = """
TODO: write your system prompt here.
Be specific about:
- The agent's role and persona
- Which tools it has and when to use each
- Output format (units, precision, language)
- What to do if it cannot answer
"""

def run_my_agent(query: str) -> str:
    """Convenience wrapper for your agent."""
    return react_agent(
        query=query,
        tools=my_agent_tools,
        system_prompt=MY_SYSTEM_PROMPT,
        max_steps=10,
        verbose=True,
    )

In [ ]:
# Query 1 — normal case
q1 = "TODO"
print(f"Query 1: {q1}")
r1 = run_my_agent(q1)
print(f"Answer: {r1}\n")

In [ ]:
# Query 2 — multi-step
q2 = "TODO"
print(f"Query 2: {q2}")
r2 = run_my_agent(q2)
print(f"Answer: {r2}\n")

In [ ]:
# Query 3 — edge case
q3 = "TODO"
print(f"Query 3: {q3}")
r3 = run_my_agent(q3)
print(f"Answer: {r3}\n")

In [ ]:
# Query 4 — no tool needed
q4 = "TODO"
print(f"Query 4: {q4}")
r4 = run_my_agent(q4)
print(f"Answer: {r4}\n")

In [ ]:
# Query 5 — your choice
q5 = "TODO"
print(f"Query 5: {q5}")
r5 = run_my_agent(q5)
print(f"Answer: {r5}\n")

### 4.4 — ✏️ Evaluation

Fill in the table and answer the questions.

**Evaluation table:**

| Query | Tools called | Correct? | Notes |
|-------|-------------|----------|-------|
| Q1 | | ✅/❌ | |
| Q2 | | ✅/❌ | |
| Q3 | | ✅/❌ | |
| Q4 | | ✅/❌ | |
| Q5 | | ✅/❌ | |

**Questions:**

1. What is the most important thing you learned about writing effective tool docstrings?
   > *Your answer:*

2. Where did your agent fail or surprise you? What would you change?
   > *Your answer:*

3. How does this ReAct agent relate to the four agentic patterns from CM1?
   > *Your answer:*

4. In TP2 your agent will retrieve information from documents (RAG) instead of calling Python functions.  
   What do you think will change in the loop? What stays the same?
   > *Your answer:*

---
## What comes next

| Session | Date | You will build |
|---------|------|----------------|
| TP2 | Nov 10–13 | RAG pipeline — retrieval as a tool, ChromaDB, RAGAS evaluation |
| TP3+TP4 | Nov 18 | Multi-agent system with CrewAI + intro to LangGraph |
| EDT1–4 | Nov 23 → Dec 8 | Your group project — agent on a real business problem |
| Soutenance | Dec 9 | Demo + Q&A |
| QCM | Dec 10 | 1h — key concepts from the module |